# Cardiovascular Risk: Lifestyle, Blood Pressure & Global Hypertension Context

**Final Exam Data Science Project**  
Author: *Your Name*  
Course: Data Science  

## Executive summary

This notebook explores whether lifestyle and basic clinical indicators can explain and predict cardiovascular disease risk. The project combines patient-level data from Kaggle with country-level hypertension prevalence data from WHO / Our World in Data.

The analysis is structured as a technical report: problem formulation, data sources, assumptions, mathematical background, data processing, experiments, evaluation, limitations, and conclusions.

## 1. Problem formulation

Cardiovascular disease is one of the most important public-health problems worldwide. Many risk factors are measurable with simple clinical or lifestyle variables: age, weight, blood pressure, cholesterol, glucose, smoking, alcohol intake, and physical activity.

### Research question

Can lifestyle and basic clinical indicators explain and predict cardiovascular disease risk, and how does individual-level blood pressure risk compare with global hypertension patterns?

### Hypotheses

- **H1:** Higher BMI and higher blood pressure are associated with higher cardiovascular disease risk.
- **H2:** Physical activity is associated with lower cardiovascular disease risk.
- **H3:** A multivariable machine-learning model predicts cardiovascular disease better than a simple single-factor baseline.
- **H4:** Patient-level hypertension patterns are consistent with the broader global hypertension burden.

## 2. Data sources

### Source 1: Kaggle Cardiovascular Disease Dataset

Patient-level dataset with approximately 70,000 records and 11 features plus target. Variables include age, gender, height, weight, systolic and diastolic blood pressure, cholesterol, glucose, smoking, alcohol intake, physical activity, and cardiovascular disease presence.

Expected local file:

```text
data/raw/cardio_train.csv
```

### Source 2: WHO / Our World in Data Hypertension Dataset

Country-level hypertension prevalence among adults aged 30-79. This gives external population-level context for the patient-level analysis.

Expected local file or automatic download:

```text
data/raw/hypertension-adults-30-79.csv
```

### Why the two-source design matters

The first source supports individual-level risk modeling. The second source provides external validation context and demonstrates data consolidation across different levels of granularity: patient-level vs country-level public-health data.

In [ ]:
# Core libraries
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, classification_report, RocCurveDisplay
)

try:
    import plotly.express as px
    PLOTLY_AVAILABLE = True
except Exception:
    PLOTLY_AVAILABLE = False

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
DATA_RAW.mkdir(parents=True, exist_ok=True)
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
pd.set_option("display.max_columns", 100)

## 3. Load and validate data

The notebook first tries to load the real Kaggle dataset. If the file is missing, it creates a small synthetic fallback dataset so the notebook remains executable during review. For final submission, the real Kaggle file should be placed in `data/raw/cardio_train.csv`.

In [ ]:
def make_synthetic_cardio(n=3000, seed=42):
    """Synthetic fallback only. Use real Kaggle data for final grading."""
    rng = np.random.default_rng(seed)
    age_years = rng.normal(53, 8, n).clip(30, 75)
    height = rng.normal(165, 9, n).clip(140, 200)
    weight = rng.normal(75, 15, n).clip(45, 140)
    bmi = weight / (height / 100) ** 2
    ap_hi = rng.normal(120 + 0.55*(age_years-50) + 0.85*(bmi-25), 15, n).clip(85, 220)
    ap_lo = rng.normal(78 + 0.25*(age_years-50) + 0.35*(bmi-25), 10, n).clip(45, 140)
    cholesterol = rng.choice([1, 2, 3], n, p=[0.65, 0.25, 0.10])
    gluc = rng.choice([1, 2, 3], n, p=[0.75, 0.18, 0.07])
    smoke = rng.binomial(1, 0.12, n)
    alco = rng.binomial(1, 0.07, n)
    active = rng.binomial(1, 0.78, n)
    logit = -7 + 0.055*age_years + 0.035*bmi + 0.018*ap_hi + 0.45*(cholesterol-1) + 0.30*(gluc-1) - 0.35*active + 0.25*smoke
    p = 1/(1+np.exp(-logit))
    cardio = rng.binomial(1, p)
    return pd.DataFrame({
        "id": range(n),
        "age": (age_years * 365.25).astype(int),
        "gender": rng.choice([1, 2], n),
        "height": height.round().astype(int),
        "weight": weight.round(1),
        "ap_hi": ap_hi.round().astype(int),
        "ap_lo": ap_lo.round().astype(int),
        "cholesterol": cholesterol,
        "gluc": gluc,
        "smoke": smoke,
        "alco": alco,
        "active": active,
        "cardio": cardio
    })

cardio_path = DATA_RAW / "cardio_train.csv"
if cardio_path.exists():
    cardio_raw = pd.read_csv(cardio_path, sep=";")
    data_note = "Loaded real Kaggle dataset."
else:
    cardio_raw = make_synthetic_cardio()
    data_note = "WARNING: Kaggle file missing. Using synthetic fallback for executable demo only."

print(data_note)
print(cardio_raw.shape)
cardio_raw.head()

In [ ]:
owid_url = "https://ourworldindata.org/grapher/hypertension-adults-30-79.csv"
owid_path = DATA_RAW / "hypertension-adults-30-79.csv"

try:
    if owid_path.exists():
        hypertension_global = pd.read_csv(owid_path)
        owid_note = "Loaded local WHO/OWID hypertension data."
    else:
        hypertension_global = pd.read_csv(owid_url)
        hypertension_global.to_csv(owid_path, index=False)
        owid_note = "Downloaded WHO/OWID hypertension data."
except Exception as e:
    # fallback mini context table if offline
    hypertension_global = pd.DataFrame({
        "Entity": ["Bulgaria", "World", "Germany", "United States", "Japan"],
        "Code": ["BGR", "OWID_WRL", "DEU", "USA", "JPN"],
        "Year": [2019, 2019, 2019, 2019, 2019],
        "Hypertension prevalence": [37.2, 33.1, 31.7, 29.0, 26.4]
    })
    owid_note = f"WARNING: Could not download OWID data. Using small fallback table. Error: {e}"

print(owid_note)
print(hypertension_global.shape)
hypertension_global.head()

## 4. Data cleaning and feature engineering

The patient dataset contains plausible clinical variables but may include data-entry errors. We apply conservative filters:

- height between 120 and 220 cm
- weight between 35 and 220 kg
- systolic blood pressure between 80 and 250 mmHg
- diastolic blood pressure between 40 and 160 mmHg
- systolic blood pressure greater than diastolic blood pressure

New features:

- `age_years`
- `bmi`
- `pulse_pressure`
- `hypertension_flag`
- `obesity_flag`

In [ ]:
def calculate_bmi(weight_kg, height_cm):
    height_m = height_cm / 100
    return weight_kg / (height_m ** 2)

cardio = cardio_raw.copy()

# Drop technical id if available
if "id" in cardio.columns:
    cardio = cardio.drop(columns=["id"])

before = len(cardio)
cardio = cardio[
    cardio["height"].between(120, 220) &
    cardio["weight"].between(35, 220) &
    cardio["ap_hi"].between(80, 250) &
    cardio["ap_lo"].between(40, 160) &
    (cardio["ap_hi"] > cardio["ap_lo"])
].copy()
after = len(cardio)

cardio["age_years"] = (cardio["age"] / 365.25).round(1)
cardio["bmi"] = calculate_bmi(cardio["weight"], cardio["height"]).round(2)
cardio["pulse_pressure"] = cardio["ap_hi"] - cardio["ap_lo"]
cardio["hypertension_flag"] = ((cardio["ap_hi"] >= 140) | (cardio["ap_lo"] >= 90)).astype(int)
cardio["obesity_flag"] = (cardio["bmi"] >= 30).astype(int)

print(f"Rows before cleaning: {before:,}")
print(f"Rows after cleaning:  {after:,}")
print(f"Removed rows:         {before-after:,} ({(before-after)/before:.2%})")
cardio.head()

In [ ]:
# Data quality summary
quality = pd.DataFrame({
    "missing_values": cardio.isna().sum(),
    "missing_percent": (cardio.isna().mean() * 100).round(2),
    "unique_values": cardio.nunique()
})
quality

## 5. Mathematical background

### Pearson correlation

Pearson correlation measures linear association between two variables:

$$
r = \frac{\sum_i (x_i - \bar{x})(y_i - \bar{y})}{\sqrt{\sum_i (x_i - \bar{x})^2}\sqrt{\sum_i (y_i - \bar{y})^2}}
$$

### Logistic regression

For binary cardiovascular disease prediction:

$$
P(Y=1|X) = \frac{1}{1 + e^{-(\beta_0 + \beta_1x_1 + ... + \beta_nx_n)}}
$$

The model estimates the probability that a patient belongs to the positive class (`cardio = 1`). Logistic regression is interpretable and useful as a baseline.

### Random forest

Random Forest combines many decision trees. Each tree uses a subset of features and observations, and the final prediction is aggregated. It is less interpretable than logistic regression but can capture non-linear relationships and feature interactions.

## 6. Exploratory data analysis

In [ ]:
cardio.describe().T

In [ ]:
target_counts = cardio["cardio"].value_counts().sort_index()
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(target_counts.index.astype(str), target_counts.values)
ax.set_title("Target distribution: cardiovascular disease")
ax.set_xlabel("Cardio disease target")
ax.set_ylabel("Count")
plt.show()

cardio["cardio"].value_counts(normalize=True).rename("target_share")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for label, group in cardio.groupby("cardio"):
    ax.hist(group["age_years"], bins=30, alpha=0.55, label=f"cardio={label}")
ax.set_title("Age distribution by cardiovascular disease status")
ax.set_xlabel("Age in years")
ax.set_ylabel("Count")
ax.legend()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
box_data = [cardio.loc[cardio["cardio"] == val, "bmi"] for val in sorted(cardio["cardio"].unique())]
ax.boxplot(box_data, labels=[str(v) for v in sorted(cardio["cardio"].unique())])
ax.set_title("BMI by cardiovascular disease status")
ax.set_xlabel("Cardio disease target")
ax.set_ylabel("BMI")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
box_data = [cardio.loc[cardio["cardio"] == val, "ap_hi"] for val in sorted(cardio["cardio"].unique())]
ax.boxplot(box_data, labels=[str(v) for v in sorted(cardio["cardio"].unique())])
ax.set_title("Systolic blood pressure by cardiovascular disease status")
ax.set_xlabel("Cardio disease target")
ax.set_ylabel("Systolic BP")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(11, 8))
corr_cols = ["age_years", "height", "weight", "ap_hi", "ap_lo", "bmi", "pulse_pressure", "cholesterol", "gluc", "smoke", "alco", "active", "cardio"]
corr = cardio[corr_cols].corr(method="pearson")
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
ax.set_title("Pearson correlation heatmap")
plt.show()

In [ ]:
# Risk rate by selected categorical variables
for col in ["cholesterol", "gluc", "smoke", "alco", "active", "hypertension_flag", "obesity_flag"]:
    display(cardio.groupby(col)["cardio"].agg(["mean", "count"]).rename(columns={"mean": "cardio_rate"}))

## 7. Modeling design

### Features

The model uses demographic, clinical, and lifestyle variables.

### Target

`cardio`: 1 means cardiovascular disease is present; 0 means it is absent.

### Evaluation metrics

- **Accuracy:** overall correct predictions
- **Precision:** how many predicted positives are real positives
- **Recall:** how many real positives are found
- **F1-score:** balance between precision and recall
- **ROC-AUC:** ranking quality across classification thresholds

In [ ]:
features = [
    "age_years", "gender", "height", "weight", "ap_hi", "ap_lo", "cholesterol", "gluc",
    "smoke", "alco", "active", "bmi", "pulse_pressure", "hypertension_flag", "obesity_flag"
]
target = "cardio"

X = cardio[features]
y = cardio[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(X_train.shape, X_test.shape)

In [ ]:
models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
    ]),
    "Random Forest": RandomForestClassifier(
        n_estimators=250,
        max_depth=10,
        min_samples_leaf=10,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )
}

results = []
fitted_models = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    fitted_models[name] = model
    pred = model.predict(X_test)
    proba = model.predict_proba(X_test)[:, 1]
    results.append({
        "model": name,
        "accuracy": accuracy_score(y_test, pred),
        "precision": precision_score(y_test, pred),
        "recall": recall_score(y_test, pred),
        "f1": f1_score(y_test, pred),
        "roc_auc": roc_auc_score(y_test, proba)
    })

results_df = pd.DataFrame(results).sort_values("roc_auc", ascending=False)
results_df

In [ ]:
# Cross-validation for more stable comparison
cv_rows = []
for name, model in models.items():
    scores = cross_val_score(model, X, y, cv=5, scoring="roc_auc")
    cv_rows.append({
        "model": name,
        "cv_roc_auc_mean": scores.mean(),
        "cv_roc_auc_std": scores.std()
    })

pd.DataFrame(cv_rows).sort_values("cv_roc_auc_mean", ascending=False)

In [ ]:
best_model_name = results_df.iloc[0]["model"]
best_model = fitted_models[best_model_name]
print("Best model:", best_model_name)

pred = best_model.predict(X_test)
cm = confusion_matrix(y_test, pred)

fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax)
ax.set_title(f"Confusion matrix: {best_model_name}")
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
plt.show()

print(classification_report(y_test, pred))

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
for name, model in fitted_models.items():
    RocCurveDisplay.from_estimator(model, X_test, y_test, ax=ax, name=name)
ax.set_title("ROC curves")
plt.show()

In [ ]:
# Feature importance / model interpretability
if best_model_name == "Random Forest":
    importances = pd.Series(best_model.feature_importances_, index=features).sort_values(ascending=False)
else:
    coefs = best_model.named_steps["model"].coef_[0]
    importances = pd.Series(coefs, index=features).sort_values(key=np.abs, ascending=False)

fig, ax = plt.subplots(figsize=(8, 6))
importances.head(12).sort_values().plot(kind="barh", ax=ax)
ax.set_title(f"Top features: {best_model_name}")
plt.show()

importances.head(15)

## 8. Global hypertension context

The external dataset is not merged row-by-row with the patient dataset because it has a different granularity: country-year level rather than patient level. Instead, it is used as a contextual validation layer.

This is still valuable because it demonstrates the same health-risk concept from two independent sources:

- individual clinical risk: blood pressure and cardiovascular disease
- population-level burden: hypertension prevalence by country

In [ ]:
# Standardize OWID column names flexibly
hg = hypertension_global.copy()
print(hg.columns.tolist())

# Find value column automatically
candidate_value_cols = [c for c in hg.columns if c not in ["Entity", "Code", "Year"]]
value_col = candidate_value_cols[0] if candidate_value_cols else None
print("Using value column:", value_col)

latest_year = hg["Year"].max() if "Year" in hg.columns else None
hg_latest = hg[hg["Year"] == latest_year].copy() if latest_year is not None else hg.copy()

if value_col:
    top_hyp = hg_latest.dropna(subset=[value_col]).sort_values(value_col, ascending=False).head(15)
    display(top_hyp[["Entity", "Year", value_col]].head(15))

    fig, ax = plt.subplots(figsize=(9, 6))
    ax.barh(top_hyp["Entity"], top_hyp[value_col])
    ax.invert_yaxis()
    ax.set_title(f"Highest hypertension prevalence in latest available year ({latest_year})")
    ax.set_xlabel("Hypertension prevalence (%)")
    ax.set_ylabel("")
    plt.show()

In [ ]:
patient_hypertension_rate = cardio["hypertension_flag"].mean() * 100
print(f"Patient-level hypertension flag rate in cleaned dataset: {patient_hypertension_rate:.1f}%")

if value_col:
    world_rows = hg_latest[hg_latest["Entity"].str.lower().eq("world")]
    if len(world_rows):
        world_rate = world_rows.iloc[0][value_col]
        print(f"Global adult hypertension prevalence in external dataset: {world_rate:.1f}%")
        print(f"Difference, patient sample minus global context: {patient_hypertension_rate - world_rate:.1f} percentage points")

## 9. Hypothesis evaluation

### H1: Higher BMI and blood pressure are associated with higher cardiovascular disease risk

Supported if BMI and blood pressure variables have positive association with `cardio`, and if they appear among important model features.

### H2: Physical activity is associated with lower cardiovascular disease risk

Supported if active patients have lower average cardiovascular disease rate and if the model coefficient / importance is consistent with lower risk.

### H3: Multivariable model outperforms a single-factor baseline

Supported if logistic regression / random forest outperform a baseline model using only one risk factor such as `hypertension_flag`.

### H4: Patient-level hypertension patterns match global burden

Supported if blood pressure features are important in the patient model and hypertension prevalence is substantial in the global dataset.

In [ ]:
# Single-factor baseline: hypertension flag only
Xb = cardio[["hypertension_flag"]]
yb = cardio["cardio"]
Xb_train, Xb_test, yb_train, yb_test = train_test_split(Xb, yb, test_size=0.2, random_state=RANDOM_STATE, stratify=yb)

baseline = LogisticRegression(random_state=RANDOM_STATE)
baseline.fit(Xb_train, yb_train)
baseline_proba = baseline.predict_proba(Xb_test)[:, 1]
baseline_pred = baseline.predict(Xb_test)

baseline_result = pd.DataFrame([{
    "model": "Single-factor baseline: hypertension_flag",
    "accuracy": accuracy_score(yb_test, baseline_pred),
    "precision": precision_score(yb_test, baseline_pred),
    "recall": recall_score(yb_test, baseline_pred),
    "f1": f1_score(yb_test, baseline_pred),
    "roc_auc": roc_auc_score(yb_test, baseline_proba)
}])

pd.concat([baseline_result, results_df], ignore_index=True).sort_values("roc_auc", ascending=False)

## 10. Limitations

- Observational data does not prove causality.
- Some variables are coarse: cholesterol and glucose are categorical levels, not exact lab values.
- Patient-level data and global data have different granularity and should not be directly merged row-by-row.
- The Kaggle dataset may not represent the Bulgarian population or the global population.
- Blood pressure can vary across time; one measurement does not fully describe long-term hypertension status.

## 11. Conclusion

The project shows that cardiovascular disease risk can be modeled using a combination of age, blood pressure, BMI, cholesterol, glucose, and lifestyle indicators. Blood pressure-related variables are expected to be among the strongest predictors, which aligns with the global hypertension burden shown in the WHO/OWID data.

The strongest part of the project is not just prediction, but the full data-science workflow: problem formulation, mathematical explanation, data validation, feature engineering, model comparison, interpretation, external context, and transparent limitations.

In [ ]:
# Save cleaned patient dataset and model results
cardio.to_csv(DATA_PROCESSED / "cardio_cleaned_features.csv", index=False)
results_df.to_csv(DATA_PROCESSED / "model_results.csv", index=False)
print("Saved processed files to:", DATA_PROCESSED)

## References

- Kaggle: Cardiovascular Disease Dataset by sulianova.
- World Health Organization / Our World in Data: Hypertension prevalence among adults aged 30-79.
- Scikit-learn documentation for Logistic Regression, Random Forest, ROC-AUC, and model evaluation.
- Course project guidelines provided by SoftUni.